In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random

In [2]:
conn = sqlite3.connect("./my_database.db")

In [3]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [4]:
test_vendor = vendor_ids.iloc[2]['vendor_id']
test_vendor

'179994ef-ab52-4434-be8c-1a46dbc79b54'

In [5]:
def unassigned_work_orders(vendor_id):
    # df = pd.read_sql(
    #     f"""
    #     select count(*) as unassigned_work_orders
    #     from work_orders
    #     where assigned_vendor='{vendor_id}'"""
    # , con=conn)
    df = pd.read_sql(
        f"""
        select work_order_id, count(*) as "unassigned tickets"
        from ticket t
        join work_orders wo
        using (work_order_id)
        where t.status = 'unassigned' and wo.assigned_vendor = '{vendor_id}'
        group by 1;
        """, con=conn
    )
    return {'vendor_id': vendor_id, 'unassigned tickets': df['unassigned tickets'].sum()}

In [6]:
unassigned_work_orders(test_vendor)

{'vendor_id': '179994ef-ab52-4434-be8c-1a46dbc79b54',
 'unassigned tickets': np.int64(72)}

In [7]:
def pending_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and (status = 'in progress' or status = 'assigned')
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [8]:
pending_tickets(test_vendor)

{'vendor_id': '179994ef-ab52-4434-be8c-1a46dbc79b54',
 'active tickets': np.int64(411)}

In [9]:
df = pd.read_sql(
    f"""select * 
    from ticket 
    where vendor_id='{test_vendor}' and completed_at is not null and created_at > date('now', '-3 months')"""
    , con=conn, parse_dates=['assigned_at', 'completed_at'])

In [10]:
df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.start_time)

In [11]:
complete_by_week = df.groupby('completion_week')['ticket_id'].count().reset_index().sort_values(by='ticket_id', ascending=False)
complete_by_week.rename(columns={'ticket_id': 'ticket_count'}, inplace=True)
complete_by_week

,completion_week,ticket_count
3,2026-02-02,9
6,2026-03-09,6
5,2026-03-02,6
4,2026-02-09,6
7,2026-03-23,5
2,2026-01-26,1
0,2026-01-12,1
1,2026-01-19,1
